In [1]:
import numpy as np


def generate_annulus_point_cloud(
    n, mean_radius=2.0, std_radius=0.3, noise_box=6.0, seed=None
):
    """
    Generate a point cloud of n points consisting of:
    - 90% points sampled from a Gaussian annulus
    - 10% points sampled uniformly from a square

    Parameters:
        n (int): Total number of points
        mean_radius (float): Mean radius for annulus
        std_radius (float): Std deviation for annulus
        noise_box (float): Half side length of square [-noise_box, noise_box]^2
        seed (int or None): Random seed for reproducibility

    Returns:
        X (np.ndarray): Array of shape (n, 2)
    """
    if seed is not None:
        np.random.seed(seed)

    n_annulus = int(0.9 * n)
    n_noise = n - n_annulus

    # Sample annulus points
    radii = np.random.normal(loc=mean_radius, scale=std_radius, size=n_annulus)
    angles = np.random.uniform(low=0, high=2 * np.pi, size=n_annulus)
    x_annulus = np.stack((radii * np.cos(angles), radii * np.sin(angles)), axis=1)

    # Sample noise points
    x_noise = np.random.uniform(low=-noise_box, high=noise_box, size=(n_noise, 2))

    # Combine
    X = np.vstack((x_annulus, x_noise))
    return X


points = generate_annulus_point_cloud(10, seed=42)
np.savetxt("points.txt", points)

- degree: degree-Rip filtration (x-axis: negative vertex degree, y-axis: float distance)
- ball_density: ball_density-Rips filtration (x-axis: negative vertex degree, y-axis: float distance)
- degree_rational: degree-Rips filtration (x-axis: negative ball density, y-axis: rational distance)
- ball_density_rational: ball_density-Rips filtration (x-axis: negative ball density, y-axis: rational distance)
- firep: a general bi-filtration representation

In [2]:
import abmph

# build a bi-filtration from a point cloud
bifi = abmph.BiFiltration(
    filtration_type="degree_rational",  # filtration type can be "degree", "degree_rational", "ball_density", "ball_density_rational", "firep"
    path="points.txt",
)

In [3]:
# You can see the size of the graph
bifi.ggraph.get_nedges(), bifi.ggraph.get_nvertices()

(349, 100)

In [4]:
# get the x and y coordinates of the grade table
# in degree-rips filtration, the x-coordinates are the degrees of the simplices
# the y-coordinates are the distance
bifi.get_x_coords()

[-9, -8, -7, -6, -5, -4, -3, -2, -1, 0]

In [5]:
# we deal with the rational numbers by converting them to strings
bifi.get_y_coords()

['0',
 '3877861/500000000',
 '9684753/100000000',
 '1315771/10000000',
 '1523463/10000000',
 '478249/2000000',
 '1234401/5000000',
 '567847/2000000',
 '2672053/5000000',
 '2698321/5000000',
 '6312581/10000000',
 '3356207/5000000',
 '4117939/5000000',
 '8702011/10000000',
 '1022547/1000000',
 '44373/40000',
 '1117081/1000000',
 '288531/250000',
 '157709/125000',
 '1269427/1000000',
 '641189/500000',
 '645067/500000',
 '55169/40000',
 '1386981/1000000',
 '1393249/1000000',
 '350251/250000',
 '764629/500000',
 '813053/500000',
 '423447/250000',
 '478409/250000',
 '120087/62500',
 '1932913/1000000',
 '1940669/1000000',
 '540129/250000',
 '2399459/1000000',
 '2496307/1000000',
 '1275903/500000',
 '2648653/1000000',
 '2683383/1000000',
 '278023/100000',
 '3030717/1000000',
 '397883/125000',
 '3223047/1000000',
 '3314641/1000000',
 '663979/200000',
 '770861/200000']

In [6]:
# then you can compute the MPH0 from the graded graph
graded_graph = bifi.ggraph
res = abmph.compute_MPH0(graded_graph)

In [7]:
# the result is a dictionary with the following keys:
# - b_0: the 0-th betti number of H_0(G)
# - b_1: the 1-st betti number of H_0(G)
# - b_2: the 2-nd betti number of H_0(G)
# - b_0_1: the 0-th betti number of H_1(G)
# - M: minimal presentation matrix
res.keys()

dict_keys(['b_0', 'b_1', 'b_2', 'b_0_1', 'M'])

You can also directly pass into an Numpy array

In [8]:
bifi_2 = abmph.BiFiltration(
    filtration_type="degree_rational",  # filtration type can be "degree", "degree_rational", "ball_density", "ball_density_rational", "firep"
    points=points,
)

bifi_2.ggraph.get_nedges(), bifi_2.ggraph.get_nvertices()

(346, 100)